# MCP Demo with Multiple Tool Servers

This notebook demonstrates how to use the Model Context Protocol (MCP) with multiple dedicated servers. We'll connect to both an OS Tool server and an Employee Information tool server, using a language model to query local files and employee data.

## Key Components

1. **Azure OpenAI Integration**: Using Azure's OpenAI service to power our agent
2. **Multi-Server MCP Setup**: Connecting to multiple tool servers simultaneously
3. **OS Tool Server**: For file system operations and local file content retrieval
4. **Employee Info Server**: For employee-related information queries
5. **Agent Creation**: Building a ReAct agent that can use tools from multiple MCP servers
6. **Cross-Server Query Execution**: Demonstrating how agents can seamlessly use tools from different servers

## Required Library Imports

The following cell imports all necessary libraries for working with multiple MCP servers, Azure OpenAI, and LangGraph.

In [5]:
from langchain_openai import AzureChatOpenAI
import random
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
import time
from langchain_mcp_adapters.client import MultiServerMCPClient
from dotenv import load_dotenv
import os
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import ToolMessage, AIMessage, HumanMessage, SystemMessage

## Azure OpenAI Configuration

Setting up the Azure OpenAI client with appropriate API credentials and parameters.

In [6]:
load_dotenv()
AZURE_API_VERSION = os.getenv("AZURE_API_VERSION_ENV")
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT_ENV")
AZURE_API_KEY = os.getenv("AZURE_API_KEY_ENV")
memory = MemorySaver()

def llm():
    model = AzureChatOpenAI(
        api_version= AZURE_API_VERSION,
        azure_endpoint= AZURE_ENDPOINT,
        api_key= AZURE_API_KEY,
        azure_deployment="gpt-4o",
        verbose=True
    )
    return model

azure_api_client = llm()
# Setting up the Azure OpenAI model and memory saver for multi-server MCP operations
# Note: In a production environment, store these keys securely using proper secret management

## Connecting to the OS Tool MCP Server

In this section, we establish a connection to the OS Tool MCP Server (running on port 8003). This server provides tools for file system operations including searching for files, reading file contents, and getting file information from the local Desktop directory.

In [14]:
# Connecting to the Employee Information server using MCP
# The server should be running on localhost:8001 with SSE transport
async with MultiServerMCPClient(
    {
        "os_tool_server": {
            "url": "http://localhost:8003/sse",
            "transport": "sse",
        }
    }
) as client:
    tools = client.get_tools()

## Displaying Available Tools

Let's examine the available tools provided by the OS Tool MCP server.

In [15]:
tools

[StructuredTool(name='get_absolute_path', description='\n    Searches for all directories with the specified folder name within the\n    "/Users/L036202/Desktop/" directory tree, excluding certain system and hidden folders.\n    \n    Args:\n    \n        folder_name (str): The name of the folder to search for.\n        \n    Returns:\n        list: A list of absolute paths to directories matching the folder_name.\n        \n    Notes:\n        - Excludes directories named \'.env\', \'.venv\', \'__pycache__\', and any directories\n          starting with \'.\' or \'_\'.\n        - Searches recursively through the entire Desktop directory tree.\n    ', args_schema={'properties': {'folder_name': {'default': None, 'title': 'Folder Name', 'type': 'string'}, 'file_name': {'default': None, 'title': 'File Name', 'type': 'string'}}, 'title': 'get_absolute_pathArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.c

## Creating the LangChain Agent

Now, let's create a LangChain agent. Our agent uses the OpenAI ChatOpenAI model and has access to tools from both MCP servers:
- **OS Tool Server**: File system operations (list files, read content, file info)
- **Employee Info Server**: HR-related queries (employee search, department info, role details)

The agent can intelligently route tool calls to the appropriate server based on the user's query.

In [16]:
session_id = "test_001"
config = {"configurable": {"thread_id": session_id}}
async with MultiServerMCPClient(
    {
        "os_tool_server": {
            "url": "http://localhost:8003/sse",
            "transport": "sse",
        }
    }
) as client:
    agent = create_react_agent(azure_api_client, client.get_tools(), checkpointer= memory)
    response = await agent.ainvoke({"messages": "What are the contents available in Wellness.txt"}, config= config)

## Agent Response

Examining the response from our agent

In [17]:
response

{'messages': [HumanMessage(content='What are the contents available in Wellness.txt', additional_kwargs={}, response_metadata={}, id='46bcccf5-b905-44e2-9451-0e97dacd5c71'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_J4zr51qcF7DN5puuE2gA1ALR', 'function': {'arguments': '{"file_name":"Wellness.txt"}', 'name': 'get_absolute_path'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 446, 'total_tokens': 465, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_e6529acef4', 'id': 'chatcmpl-ByFhVyjnuNON6Jxq0S1sL4uQLrHJ4', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': Fa

printing Agent Response in a readable way

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def print_conversation_flow(response):
    """
    Extracts and prints the conversation flow in a structured format
    showing each query, tool calls, arguments, tool messages, and AI responses.
    """
    messages = response['messages']
    
    print("=" * 60)
    print("CONVERSATION FLOW")
    print("=" * 60)

    tool_call_count = 1
    i = 0

    while i < len(messages):
        msg = messages[i]

        # Print user queries
        if isinstance(msg, HumanMessage) and msg.content:
            print(f"Query: {msg.content}")
            print("-" * 60)
            i += 1
            continue

        # Print tool call by AI
        if isinstance(msg, AIMessage) and getattr(msg, 'tool_calls', None):
            for tool_call in msg.tool_calls:
                print(f"Tool Call #{tool_call_count}:")
                print(f"  tool_called: {tool_call['name']}")
                print(f"  tool_args: {tool_call['args']}")

                # Find the corresponding ToolMessage
                tool_message = None
                if (i + 1) < len(messages):
                    next_msg = messages[i + 1]
                    if isinstance(next_msg, ToolMessage) and next_msg.tool_call_id == tool_call['id']:
                        tool_message = next_msg.content

                print(f"  tool_message: {tool_message}")
                print("-" * 40)
                tool_call_count += 1
            i += 2  # Skip tool message after AIMessage
            continue

        # Print final AI response if not a tool call
        if isinstance(msg, AIMessage) and msg.content and not getattr(msg, 'tool_calls', None):
            print(f"AI_message: {msg.content}")
            print("=" * 60)
        
        i += 1


CONVERSATION FLOW
Query: What are the contents available in Wellness.txt
------------------------------------------------------------
Tool Call #1:
  tool_called: get_absolute_path
  tool_args: {'file_name': 'Wellness.txt'}
  tool_message: /Users/L036202/Desktop/test_folder_mcp/Welness_docs/Wellness.txt
----------------------------------------
Tool Call #2:
  tool_called: get_file_info
  tool_args: {'file_path': '/Users/L036202/Desktop/test_folder_mcp/Welness_docs/Wellness.txt'}
  tool_message: {"content": "The pursuit of happiness can sometimes seem like a difficult thing. Contentment, peace, and satisfaction can elude us all too well. In times like these, mental and emotional well-being are more important than ever. They are the cornerstones of a healthy and fulfilling life. In this article, we'll take a closer look at mental and emotional health and provide actionable ways to promote wellness in our daily lives."}
----------------------------------------


## Connecting to another employee info MCP Server

## Multi-Server MCP Connection Setup

Now we'll connect to both servers simultaneously:
- **OS Tool Server** (port 8003): Handles file system operations
- **Employee Info Server** (port 8002): Manages employee data queries

This demonstrates the power of MCP's multi-server architecture where a single agent can access tools from multiple specialized servers.

In [21]:
# Multi-Server MCP Connection: Connecting to both OS Tool and Employee Info servers
# This creates a unified client that can access tools from multiple MCP servers
# 
# Server Configuration:
# - os_tool_server: localhost:8003/sse (file system operations)
# - employee_info_server: localhost:8002/sse (HR/employee data)
#
# The MultiServerMCPClient automatically handles:
# 1. Connection management to multiple servers
# 2. Tool discovery and aggregation across servers  
# 3. Routing tool calls to the appropriate server
# 4. Error handling and reconnection logic

async with MultiServerMCPClient(
    {
        "os_tool_server": {
            "url": "http://localhost:8003/sse",
            "transport": "sse",
        }, 
        "employee_info_server":{
            "url": "http://localhost:8002/sse",
            "transport": "sse",
        }

    }
) as client:
    agent = create_react_agent(azure_api_client, client.get_tools(), checkpointer= memory)
    response = await agent.ainvoke({"messages": "Who is the supervisor of David?"}, config= config)



In [22]:
response

{'messages': [HumanMessage(content='What are the contents available in Wellness.txt', additional_kwargs={}, response_metadata={}, id='46bcccf5-b905-44e2-9451-0e97dacd5c71'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_J4zr51qcF7DN5puuE2gA1ALR', 'function': {'arguments': '{"file_name":"Wellness.txt"}', 'name': 'get_absolute_path'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 446, 'total_tokens': 465, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_e6529acef4', 'id': 'chatcmpl-ByFhVyjnuNON6Jxq0S1sL4uQLrHJ4', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': Fa

## Analyzing Multi-Server Agent Response

Let's examine the raw response to understand how the agent:
- Processed the employee-related query
- Selected appropriate tools from the Employee Info Server
- Formatted the response with proper context

In [25]:
print_conversation_flow(response)

CONVERSATION FLOW
Query: What are the contents available in Wellness.txt
------------------------------------------------------------
Tool Call #1:
  tool_called: get_absolute_path
  tool_args: {'file_name': 'Wellness.txt'}
  tool_message: /Users/L036202/Desktop/test_folder_mcp/Welness_docs/Wellness.txt
----------------------------------------
Tool Call #2:
  tool_called: get_file_info
  tool_args: {'file_path': '/Users/L036202/Desktop/test_folder_mcp/Welness_docs/Wellness.txt'}
  tool_message: {"content": "The pursuit of happiness can sometimes seem like a difficult thing. Contentment, peace, and satisfaction can elude us all too well. In times like these, mental and emotional well-being are more important than ever. They are the cornerstones of a healthy and fulfilling life. In this article, we'll take a closer look at mental and emotional health and provide actionable ways to promote wellness in our daily lives."}
----------------------------------------
AI_message: The contents of 

## Conclusion

In this notebook, we've demonstrated how to use the Model Context Protocol (MCP) to create a sophisticated multi-server agent architecture. Our agent can seamlessly interact with multiple tool servers:

1. **OS Tool Server**: Provides file system operations (list directories, read file contents, get file information)
2. **Employee Info Server**: Provides HR-related functionality (employee lookup, department queries, supervisor relationships)

### Key Benefits of This Architecture:

1. **Modular Tool Development**: Each tool server can be developed and maintained independently by different teams
2. **Intelligent Tool Routing**: The agent automatically determines which server's tools to use based on query context
3. **Scalability**: Additional tool servers can be easily connected without modifying the agent code
4. **Cross-Server Capabilities**: Complex queries can leverage tools from multiple servers in a single conversation
5. **Maintainability**: Updates to individual tool servers don't require agent reconfiguration

### What We've Demonstrated:

- **Single-Server Queries**: File operations and employee lookups working independently
- **Cross-Server Intelligence**: Agent selecting appropriate tools based on query context
- **Advanced Workflows**: Complex queries that might require multiple tool servers
- **Robust Response Formatting**: Clear conversation flow printing using LangChain message types

This approach is particularly powerful for building enterprise applications where different teams maintain different services, data sources, and business logic. The MCP protocol provides a standardized way to expose these capabilities to AI agents while maintaining separation of concerns.

## References

1. [LangChain MCP Adapters](https://github.com/langchain-ai/langchain-mcp-adapters)
2. [Model Context Protocol - Tools](https://modelcontextprotocol.io/docs/concepts/tools)
3. [LangGraph MCP Agents Hands-On](https://github.com/teddynote-lab/langgraph-mcp-agents/blob/master/MCP-HandsOn-ENG.ipynb)